# OpenAI framework example (compare offers)

Follow up the readme.md to get started

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, ModelSettings
from openai import OpenAI
from pydantic import BaseModel, Field
import os
import asyncio


load_dotenv(override=True)

In [ ]:
offer_names = ["prosermo", "enpal", "arndt", "widmann"]
#offer_names = ["prosermo", "arndt"]

In [ ]:
# Generator function. Yields result to calling function
def handle_files_generator(files):
    for filename in files:
        # This is not printing the result
        yield filename + ".txt"

def read_offer_file(filename):
    with open(filename, 'r') as file:
        return file.read()

# Call generator as an iterator and print the result
for file in handle_files_generator(offer_names):
    print(file)

In [ ]:
class EnergyOffer(BaseModel):
    concerns_PV: bool = Field(description="Whether the offer is related to photovoltaics (PV) and wallbox")
    concerns_WP: bool = Field(description="Whether the offer is related to heat pumps (WP)")
    PV_offer: str = Field(description="The photovoltaic (PV) and wallbox related offer details formatted as markdown, starting with the supplier name")
    WP_offer: str = Field(description="The heat pump (WP) related offer details formatted as markdown, starting with the supplier name")

    def to_markdown(self) -> str:
        # Custom method to convert the model to a Markdown representation
        return (
            f"# PV offer: \n{self.PV_offer}\n\n"
            f"# WP offer: \n{self.WP_offer}\n\n"
        )

class OfferCompare(BaseModel):
    PV_ranking: str = Field(description="How the PV offers are ranked compared to the other offers (1 = best, 2 = second best, etc.) and why")
    WP_ranking: str = Field(description="How the WP offers are ranked compared to the other offers (1 = best, 2 = second best, etc.) and why")
    Proposal: str = Field(description="Which combination of PV and WP offers is proposed as the best option and why")

EnergyOffer.model_json_schema()

In [ ]:
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5.4-mini")
BASE_URL = os.getenv("BASE_URL", "https://api.openai.com/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

AI_client = OpenAI(base_url=BASE_URL, api_key=OPENAI_API_KEY)
settings = ModelSettings(model=MODEL_NAME, base_url=BASE_URL, api_key=OPENAI_API_KEY)

instructions = "You route the offers depending on their content (PV or WP)."
content_router_agent = Agent(name="content_router", instructions=instructions, output_type=EnergyOffer, model_settings=settings)

tasks = []

async def route_offers(offer_name):
    content = read_offer_file("../docs/" + offer_name)
    result = await Runner.run(content_router_agent, offer_name + "\n" + content)
    return result

async def run_searches():
    tasks = [route_offers(offer_name) for offer_name in handle_files_generator(offer_names)]
    results = await asyncio.gather(*tasks)
    return results

with trace("Routing offers based on content"):
    results = await run_searches()

reports = ""
offer_results = [result.final_output.to_markdown() for result in results]
for offer in offer_results:
    print(offer)
    reports += offer + "\n\n"

In [ ]:
#reports = ""
# Alternative sequential approach
if False:
    with trace("Routing offers based on content"):
        for offer_name in handle_files_generator(offer_names):
            result = await Runner.run(content_router_agent, read_offer_file("../docs/" + offer_name))
            reports = reports + "\n" + f"\nOffer: {offer_name}\n\n{result.final_output.to_markdown()}"
            #print(f"Offer: {offer_name}")
            #print(f"Routing Result: PV is {result.final_output.concerns_PV}, WP is {result.final_output.concerns_WP}")
            #print(f"PV Offer Details: {result.final_output.PV_offer}")
            #print(f"WP Offer Details: {result.final_output.WP_offer}")
    print(reports)

In [ ]:
instructions = "You compare the PV and WP offers from different suppliers based on technical, quality and economical aspects. Format your response as markdown."
compare_agent = Agent(name="compare_agent", instructions=instructions, output_type=OfferCompare, model_settings=settings)

result = await Runner.run(compare_agent, reports)
print(result.final_output)
print(result)
with open("../docs/comparison_report.md", "w", encoding="utf-8") as f:
    f.write(f"# PV ranking\n\n{result.final_output.PV_ranking}\n\n")
    f.write(f"# WP ranking\n\n{result.final_output.WP_ranking}\n\n")
    f.write(f"# Proposal\n\n{result.final_output.Proposal}\n")

